In [1]:
import numpy as np
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from rank_bm25 import BM25Okapi
import re

data_path = util.download_and_unzip( "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip","./datasets")
corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")
doc_ids = list(corpus.keys())
doc_texts = []
for doc_id in doc_ids:
    title = corpus[doc_id].get("title", "") or ""
    text  = corpus[doc_id].get("text", "")  or ""
    doc_texts.append((title + " " + text).strip())

def tokenize(text: str):
    # lowercase, keep alphanumerics, split on non-word
    return re.findall(r"[a-z0-9]+", text.lower())

doc_tokens_list = [tokenize(t) for t in doc_texts]
bm25 = BM25Okapi(doc_tokens_list)

/home/kroybal/miniconda3/envs/cs197-rag/lib/python3.12/site-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


  0%|          | 0/5183 [00:00<?, ?it/s]

In [2]:
import cg_rag
import cg_rag.baselines.retrievers as R
import cg_rag.models.embedding as E
import cg_rag.structures as S
embedder = E.Embedder() 
doc_vectors = embedder.embed(doc_texts, batch_size=64, show_progress=True)
segments = []
segment_doc_ids = [] # same length as segments
for i, (doc_id, text, vec) in enumerate(zip(doc_ids, doc_texts, doc_vectors)):
    seg = S.Segment(
    text=text,
    vector=vec,
    start_idx=i,
    end_idx=i+1,
    sentences=[], # optional
    internal_cost=0.0 # optional
    )
    segments.append(seg)
    segment_doc_ids.append(doc_id)
# Map python object identity -> index, so we can recover doc_id later
seg_id_to_idx = {id(seg): i for i, seg in enumerate(segments)}
doc_vecs = np.vstack([seg.vector for seg in segments]).astype(np.float32)  # shape: (N, d)
# Precompute norms once for faster cosine
doc_norms = np.linalg.norm(doc_vecs, axis=1) + 1e-12

GPU detected. Using high-fidelity embedder: all-MiniLM-L6-v2
Loading embedding model: None
Embedder ready (dim=384)


Batches:   0%|          | 0/81 [00:00<?, ?it/s]

In [3]:
def cosine(a, b, eps=1e-12):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps))

In [4]:
def embed_one(text: str) -> np.ndarray:
    return embedder.embed([text], batch_size=1, show_progress=False)[0]

In [5]:
def update_semantic_residual(r_sem, d_emb, alpha=1.0, eps=1e-12):
    d_hat = d_emb / (np.linalg.norm(d_emb) + eps)
    proj = np.dot(r_sem, d_hat) * d_hat
    return r_sem - alpha * proj

In [6]:
def update_lexical_residual(r_lex, doc_tokens, beta=1.0):
    # beta=1.0 = hard remove, beta=0.3 = soft downweight
    for t in list(r_lex.keys()):
        if t in doc_tokens:
            if beta >= 1.0:
                del r_lex[t]
            else:
                r_lex[t] *= (1.0 - beta)
                if r_lex[t] < 1e-6:
                    del r_lex[t]
    return r_lex

In [7]:
def residual_stop(r_sem, q_sem0, r_lex, lex0_sum,
                  eps_sem_ratio=0.15, eps_lex_ratio=0.10):
    sem_ratio = np.linalg.norm(r_sem) / (np.linalg.norm(q_sem0) + 1e-12)
    lex_ratio = (sum(r_lex.values()) / (lex0_sum + 1e-12)) if lex0_sum > 0 else 0.0
    return (sem_ratio < eps_sem_ratio) and (lex_ratio < eps_lex_ratio), sem_ratio, lex_ratio

In [8]:
def zscore(x, eps=1e-12):
    x = np.asarray(x)
    return (x - x.mean()) / (x.std() + eps)

In [9]:
def retrieve_next(r_sem, r_lex, *,
                  doc_vecs, doc_norms, bm25,
                  lambda_sem=0.6,
                  used_idxs=None):
    """
    Returns best document index according to hybrid score.
    - r_sem: semantic residual vector (d,)
    - r_lex: dict residual terms -> weights
    - used_idxs: set of already selected doc indices (avoid duplicates)
    """
    if used_idxs is None:
        used_idxs = set()

    # --- semantic scores (vectorized cosine) ---
    r_norm = np.linalg.norm(r_sem) + 1e-12
    sem_scores = (doc_vecs @ r_sem) / (doc_norms * r_norm)  # shape (N,)

    # --- lexical scores (BM25) using remaining residual terms as query ---
    remaining_terms = list(r_lex.keys())
    if remaining_terms:
        lex_scores = np.array(bm25.get_scores(remaining_terms), dtype=np.float32)
    else:
        lex_scores = np.zeros(len(sem_scores), dtype=np.float32)

    # --- combine ---
    sem_scores = zscore(sem_scores)
    lex_scores = zscore(lex_scores)
    scores = lambda_sem * sem_scores + (1.0 - lambda_sem) * lex_scores

    # mask out used docs
    if used_idxs:
        used = np.array(list(used_idxs), dtype=int)
        scores[used] = -1e30

    best_idx = int(np.argmax(scores))
    best_score = float(scores[best_idx])
    if best_score <= -1e20:
        return None  # everything masked out
    return best_idx

In [10]:
def iterative_retrieve(query, *, embed, tokenize,
                       doc_texts, doc_tokens_list,
                       doc_vecs, doc_norms, bm25,
                       max_steps=5, alpha=1.0, beta=1.0,
                       eps_sem_ratio=0.15, eps_lex_ratio=0.10,
                       min_gain_sem=0.05, min_gain_lex=0.05,
                       patience=2,
                       lambda_sem=0.6):
    # initial residuals
    q_sem0 = embed(query)
    r_sem = q_sem0.copy()

    q_terms = tokenize(query)
    r_lex = {t: 1.0 for t in q_terms}
    lex0_sum = sum(r_lex.values())

    chosen_idxs = []
    used_idxs = set()
    no_progress = 0

    for step in range(max_steps):
        doc_idx = retrieve_next(
            r_sem, r_lex,
            doc_vecs=doc_vecs, doc_norms=doc_norms, bm25=bm25,
            lambda_sem=lambda_sem,
            used_idxs=used_idxs
        )
        if doc_idx is None:
            break

        used_idxs.add(doc_idx)

        d_emb = doc_vecs[doc_idx]
        doc_tokens = set(doc_tokens_list[doc_idx])

        # gains
        gain_sem = max(0.0, cosine(r_sem, d_emb))
        if sum(r_lex.values()) > 0:
            covered = sum(w for t, w in r_lex.items() if t in doc_tokens)
            gain_lex = covered / (sum(r_lex.values()) + 1e-12)
        else:
            gain_lex = 0.0

        # accept?
        if gain_sem >= min_gain_sem or gain_lex >= min_gain_lex:
            chosen_idxs.append(doc_idx)
            r_sem = update_semantic_residual(r_sem, d_emb, alpha=alpha)
            r_lex = update_lexical_residual(r_lex, doc_tokens, beta=beta)
            no_progress = 0
        else:
            no_progress += 1

        stop, sem_ratio, lex_ratio = residual_stop(
            r_sem, q_sem0, r_lex, lex0_sum,
            eps_sem_ratio=eps_sem_ratio,
            eps_lex_ratio=eps_lex_ratio
        )
        if stop or no_progress >= patience:
            break

    return chosen_idxs

In [14]:
# pick a query id
qid = list(queries.keys())[1]
query_text = queries[qid]

chosen_idxs = iterative_retrieve(
    query_text,
    embed=embed_one,   # IMPORTANT: see note below
    tokenize=tokenize,
    doc_texts=doc_texts,
    doc_tokens_list=doc_tokens_list,
    doc_vecs=doc_vecs,
    doc_norms=doc_norms,
    bm25=bm25,
    max_steps=5,
    lambda_sem=0.6
)

# Convert indices -> doc_ids
chosen_doc_ids = [doc_ids[i] for i in chosen_idxs]
chosen_doc_ids[:10], query_text

(['14717500', '14019636', '2107238', '25571386', '39661951'],
 '1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.')

In [15]:
gold = set(qrels[qid].keys())              # relevant doc_ids for this query
retrieved = set(chosen_doc_ids)

print("Retrieved:", chosen_doc_ids)
print("Hits:", len(gold & retrieved))
print("Gold count:", len(gold))
print("Hit doc_ids:", list(gold & retrieved)[:10])

Retrieved: ['14717500', '14019636', '2107238', '25571386', '39661951']
Hits: 1
Gold count: 1
Hit doc_ids: ['14717500']


In [17]:
results = []
num_queries = len(queries)

for step in range(0, 101, 5):   # step size = 5
    lambda_sem = step / 100.0
    total_hits = 0

    for qid, query_text in queries.items():
        chosen_idxs = iterative_retrieve(
            query_text,
            embed=embed_one,
            tokenize=tokenize,
            doc_texts=doc_texts,
            doc_tokens_list=doc_tokens_list,
            doc_vecs=doc_vecs,
            doc_norms=doc_norms,
            bm25=bm25,
            max_steps=5,
            lambda_sem=lambda_sem
        )

        chosen_doc_ids = [doc_ids[idx] for idx in chosen_idxs]
        gold = set(qrels[qid].keys())
        retrieved = set(chosen_doc_ids)

        if gold & retrieved:
            total_hits += 1

    hit_rate = total_hits / num_queries
    results.append((lambda_sem, hit_rate))
    print(f"Lambda: {lambda_sem:.2f} | Hit rate: {hit_rate:.4f}")

Hit rate: 0.6333333333333333
